In [3]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import wittgenstein as lw

# --- 1. Tải tập dữ liệu Iris ---
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['target'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

X = df.drop('target', axis=1)
y = df['target']

# --- 2. Chia tập dữ liệu ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 3. Huấn luyện thuật toán RIPPER (Tự làm One-vs-Rest) ---
models = {}
classes = ['setosa', 'versicolor', 'virginica']

print("--- CÁC LUẬT PHÂN LOẠI (LEARNED RULES) ---")
for cls in classes:
    # Bước quan trọng: Tạo nhãn nhị phân. 
    # Nếu là class đang xét thì đánh True (1), ngược lại là False (0)
    y_train_binary = (y_train == cls)
    
    # Huấn luyện mô hình cho riêng class này
    clf = lw.RIPPER(random_state=42)
    clf.fit(X_train, y_train_binary)
    models[cls] = clf
    
    # In ra luật
    print(f"\nLuật để nhận diện [{cls}]:")
    clf.out_model()
    
print("-" * 50)

# --- 4 & 5. Dự đoán & Đánh giá ---
# Tạo một DataFrame để chứa xác suất dự đoán của từng class
probs = pd.DataFrame(index=X_test.index)

# Lấy xác suất dự đoán (predict_proba) từ từng mô hình
for cls, clf in models.items():
    # Cột 1 chứa xác suất của nhãn True (tức là xác suất rơi vào class đang xét)
    probs[cls] = clf.predict_proba(X_test)[:, 1]

# Lựa chọn class có xác suất cao nhất cho mỗi dòng
y_pred = probs.idxmax(axis=1)

print("\n--- ĐÁNH GIÁ MÔ HÌNH ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

--- CÁC LUẬT PHÂN LOẠI (LEARNED RULES) ---

Luật để nhận diện [setosa]:
[[petalwidth(cm)=<0.2] V
[petalwidth(cm)=0.2-0.4] V
[petallength(cm)=1.5-1.67]]

Luật để nhận diện [versicolor]:
[[petallength(cm)=4.25-4.6] V
[petallength(cm)=3.96-4.25] V
[petallength(cm)=1.67-3.96 ^ petalwidth(cm)=0.4-1.2] V
[petallength(cm)=4.6-5.0 ^ petalwidth(cm)=1.3-1.5]]

Luật để nhận diện [virginica]:
[[petallength(cm)=>5.81] V
[petallength(cm)=5.32-5.81] V
[petallength(cm)=5.0-5.32] V
[petallength(cm)=4.6-5.0 ^ sepalwidth(cm)=<2.5] V
[petalwidth(cm)=1.5-1.8 ^ sepallength(cm)=6.0-6.3]]
--------------------------------------------------

--- ĐÁNH GIÁ MÔ HÌNH ---
Accuracy: 0.9333

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                        